## FINETUNE V1.1 FOR GENERAL DATA

In [ ]:
from pathlib import Path

import sentence_transformers

PROJECT_DIR = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not locate the project directory')

MODEL_DIR = PROJECT_DIR / 'models' / 'checkpoint-40236'
DB_DIR = PROJECT_DIR / 'database' / 'general_triplet.db'

if not MODEL_DIR.is_dir():
    raise FileNotFoundError(f'Model directory not found: {MODEL_DIR}')
if not DB_DIR.is_file():
    raise FileNotFoundError(f'Database file not found: {DB_DIR}')

In [ ]:
import sqlite3


def get_num_records():
    query = """
        SELECT COUNT(*)
        FROM general_triplet
        WHERE anchor IS NOT NULL
          AND positive IS NOT NULL
          AND hard_negative IS NOT NULL
    """
    with sqlite3.connect(DB_DIR) as con:
        return con.execute(query).fetchone()[0]


def get_data(batch=10_000):
    con = sqlite3.connect(DB_DIR)
    con.row_factory = sqlite3.Row

    query = """
        SELECT anchor, positive, hard_negative
        FROM general_triplet
        WHERE anchor IS NOT NULL
          AND positive IS NOT NULL
          AND hard_negative IS NOT NULL
        ORDER BY RANDOM()
    """
    cursor = con.cursor()
    cursor.execute(query)

    while True:
        rows = cursor.fetchmany(batch)
        if not rows:
            break

        for row in rows:
            yield {
                'anchor': 'query: ' + row['anchor'],
                'positive': 'passage: ' + row['positive'],
                'hard_negative': 'passage: ' + row['hard_negative'],
            }

    cursor.close()
    con.close()

In [ ]:
import math
import torch
from datasets import Features, IterableDataset, Value
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

model = SentenceTransformer(str(MODEL_DIR))
loss = MultipleNegativesRankingLoss(model)

train_features = Features({
    'anchor': Value('string'),
    'positive': Value('string'),
    'hard_negative': Value('string'),
})
train_dataset = IterableDataset.from_generator(get_data, features=train_features)

num_records = get_num_records()
batch_size = 32
epochs = 2
gradient_accumulation_steps = 1

num_gpus = max(torch.cuda.device_count(), 1)
effective_batch_size = batch_size * num_gpus * gradient_accumulation_steps
steps_per_epoch = math.ceil(num_records / effective_batch_size)
max_steps = steps_per_epoch * epochs

print('Records:', num_records)
print('GPUs:', num_gpus)
print('Effective batch size:', effective_batch_size)
print('Steps / epoch:', steps_per_epoch)
print('Max steps:', max_steps)

args = SentenceTransformerTrainingArguments(
    output_dir=PROJECT_DIR / 'models' / 'vietnamese-embedding-v1.1-general',
    num_train_epochs=epochs,
    max_steps=max_steps,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=100,
    save_strategy='steps',
    save_steps=5000,
    save_total_limit=2,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

In [ ]:
trainer.train()

FINAL_MODEL_DIR = PROJECT_DIR / 'models' / 'vietnamese-embedding-v1.1-general-final'
trainer.save_model(str(FINAL_MODEL_DIR))
print(f'Final model exported to: {FINAL_MODEL_DIR}')